In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

from glob import glob

import IPython.display as ipd
from tqdm import tqdm

import subprocess

plt.style.use('ggplot')

# **Converting Video Type**

In [ ]:
input_file = '/kaggle/input/datasets/robikscube/driving-video-with-object-tracking/bdd100k_videos_train_00/bdd100k/videos/train/026c7465-309f6d33.mov'
subprocess.run(['ffmpeg',
                '-y',
                '-i',
                input_file,
                '-qscale',
                '0',
                '026c7465-309f6d33.mp4',
                '-loglevel',
                'quiet']
              )

In [ ]:
!ls -GFlash --color

In [ ]:
ipd.Video('026c7465-309f6d33.mp4', width=700, embed=True)

# Open the Video and Read Metadata

In [ ]:
cap = cv2.VideoCapture('026c7465-309f6d33.mp4')

In [ ]:
cap.get(cv2.CAP_PROP_FRAME_COUNT) #number of frames

In [ ]:
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
print(f'Height {height}, Width {width}')

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'FPS : {fps:0.2f}')

In [ ]:
cap.release()

# Extracting images from video 

In [ ]:
cap = cv2.VideoCapture('026c7465-309f6d33.mp4')
ret, img = cap.read()
print(f'Returned {ret} and img of shape {img.shape}')

In [ ]:
def display_cv2_img(img, figsize=(10, 10)):
    img_ = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img_)
    ax.axis("off")

In [ ]:
display_cv2_img(img)

In [ ]:
cap.release()

# Add Annotations to Video Images

In [ ]:
labels = pd.read_csv('/kaggle/input/datasets/robikscube/driving-video-with-object-tracking/mot_labels.csv',
                     low_memory=False)
video_labels = (
    labels.query('videoName == "026c7465-309f6d33"').reset_index(drop=True).copy()
)
video_labels["video_frame"] = (video_labels["frameIndex"] * 11.9).round().astype("int")

In [ ]:
video_labels["category"].value_counts()

In [ ]:
cap = cv2.VideoCapture("026c7465-309f6d33.mp4")
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

img_idx = 0
for frame in range(n_frames):
    ret, img = cap.read()
    if ret == False:
        break
    if frame == 1035:
        break
cap.release()

In [ ]:
display_cv2_img(img)

In [ ]:
img_example = img.copy()
frame_labels = video_labels.query('video_frame == 1035')
for i, d in frame_labels.iterrows():
    pt1 = int(d['box2d.x1']), int(d['box2d.y1'])
    pt2 = int(d['box2d.x2']), int(d['box2d.y2'])
    cv2.rectangle(img_example, pt1, pt2, (0, 0, 255), 3)

display_cv2_img(img_example)

In [ ]:
color_map = {
    "car": (0, 0, 255),
    "truck": (0, 0, 100),
    "pedestrian": (255, 0, 0),
    "other vehicle": (0, 0, 150),
    "rider": (200, 0, 0),
    "bicycle": (0, 255, 0),
    "other person": (200, 0, 0),
    "trailer": (0, 150, 150),
    "motorcycle": (0, 150, 0),
    "bus": (0, 0, 100),
}

img_example = img.copy()
frame_labels = video_labels.query('video_frame == 1035')
for i, d in frame_labels.iterrows():
    pt1 = int(d['box2d.x1']), int(d['box2d.y1'])
    pt2 = int(d['box2d.x2']), int(d['box2d.y2'])
    color = color_map[d['category']]
    cv2.rectangle(img_example, pt1, pt2, color, 3)

display_cv2_img(img_example)

In [ ]:
def add_annotations(img, frame, video_labels):
    max_frame = video_labels.query("video_frame <= @frame")["video_frame"].max()
    frame_labels = video_labels.query("video_frame == @max_frame")
    for i, d in frame_labels.iterrows():
        pt1 = int(d["box2d.x1"]), int(d["box2d.y1"])
        pt2 = int(d["box2d.x2"]), int(d["box2d.y2"])
        color = color_map[d["category"]]
        img = cv2.rectangle(img, pt1, pt2, color, 3)
    return img

In [ ]:
!rm -r out_test.mp4

In [ ]:
VIDEO_CODEC = "mp4v"
fps = 59.94
width = 1280
height = 720
out = cv2.VideoWriter("out_test.mp4",
                cv2.VideoWriter_fourcc(*VIDEO_CODEC),
                fps,
                (width, height))

cap = cv2.VideoCapture("026c7465-309f6d33.mp4")
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

for frame in tqdm(range(n_frames), total=n_frames):
    ret, img = cap.read()
    if ret == False:
        break
    img = add_annotations(img, frame, video_labels)
    out.write(img)
out.release()
cap.release()

In [ ]:
!ls -GFlash -color

In [ ]:
tmp_output_path = "out_test.mp4"
output_path = "out_test_compressed.mp4"
subprocess.run(
    [
        "ffmpeg",
        "-i",
        tmp_output_path,
        "-crf",
        "18",
        "-preset",
        "veryfast",
        "-vcodec",
        "libx264",
        output_path,
        '-loglevel',
        'quiet'
    ]
)

In [ ]:
!ls -GFlash --color

In [ ]:
ipd.Video('out_test_compressed.mp4', width=600)

# Detection using YOLO tracking using Deepseek

In [ ]:
!pip install ultralytics
!pip install deep-sort-realtime

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from IPython.display import display, HTML
import os

from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

In [ ]:
VIDEO_PATH   = '/kaggle/input/datasets/robikscube/driving-video-with-object-tracking/bdd100k_videos_train_00/bdd100k/videos/train/0000f77c-cb820c98.mov'
OUTPUT_VIDEO = '/kaggle/working/tracked_output.mp4'
CONF_THRESH  = 0.4          
MAX_FRAMES   = None         
SKIP_FRAMES  = 1            


TRACK_CLASSES = {0: 'person', 1: 'bicycle', 2: 'car',
                 3: 'motorcycle', 5: 'bus', 7: 'truck'}
PALETTE = [
    (255,  56,  56), (255, 157,  151), (255, 112,  31),
    (255, 178,  29), ( 207, 210,  49), (72,  249,  10),
    ( 146, 204,  23), ( 61, 219, 134), ( 26, 147,  52),
    ( 0,  212, 187), ( 44, 153, 168), ( 0, 194, 255),
    ( 52,  69, 147), ( 100,  115, 255), (0,   24, 236),
    (132,  56, 255), (82,   0, 133), (203,  56, 255),
    (255, 149, 200), (255,  55, 199),
]

def track_color(track_id):
    return PALETTE[int(track_id) % len(PALETTE)]

In [ ]:
model   = YOLO('yolov8n.pt')
tracker = DeepSort(
    max_age           = 30,
    n_init            = 3,
    nms_max_overlap   = 1.0,
    max_cosine_distance = 0.2,
    nn_budget         = 100,
)

In [ ]:
def draw_track(frame, track_id, class_name, x1, y1, x2, y2, trail=None):
    
    color = track_color(track_id)

    #
    if trail and len(trail) > 1:
        for i in range(1, len(trail)):
            alpha = i / len(trail)
            c = tuple(int(ch * alpha) for ch in color)
            cv2.line(frame, trail[i-1], trail[i], c, 2)

    
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    
    label = f'ID{track_id} {class_name}'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)


def xyxy_to_xywh(x1, y1, x2, y2):
    return x1, y1, x2 - x1, y2 - y1

In [ ]:
def run_tracking(video_path, output_path, max_frames=None, skip=1):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, FPS, (W, H))

    
    trails      = defaultdict(list)
    TRAIL_LEN   = 30

    
    stats = {
        'total_frames'    : 0,
        'total_detections': 0,
        'unique_ids'      : set(),
        'class_counts'    : defaultdict(int),
        'fps_list'        : [],
    }

    frame_idx = 0
    preview_frames = []           

    print("▶  Starting tracking …")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_idx >= max_frames:
            break

        import time
        t0 = time.time()

        
        if frame_idx % skip == 0:
            results    = model(frame, verbose=False)[0]
            detections = []
            for box in results.boxes:
                cls  = int(box.cls[0])
                conf = float(box.conf[0])
                if conf < CONF_THRESH:
                    continue
                if TRACK_CLASSES and cls not in TRACK_CLASSES:
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                cname = TRACK_CLASSES.get(cls, model.names[cls])
                detections.append(([x1, y1, x2-x1, y2-y1], conf, cname))

            tracks = tracker.update_tracks(detections, frame=frame)
        

       
        active_ids = 0
        for track in tracks:
            if not track.is_confirmed():
                continue
            tid   = int(track.track_id)
            cname = track.det_class or 'object'
            l, t, r, b = map(int, track.to_ltrb())

           
            l, t = max(0, l), max(0, t)
            r, b = min(W, r), min(H, b)

            cx, cy = (l + r) // 2, (t + b) // 2  # centroid

           
            trails[tid].append((cx, cy))
            if len(trails[tid]) > TRAIL_LEN:
                trails[tid].pop(0)

            draw_track(frame, tid, cname, l, t, r, b, trail=trails[tid])

            
            stats['unique_ids'].add(tid)
            stats['class_counts'][cname] += 1
            active_ids += 1

        stats['total_detections'] += active_ids
        stats['total_frames']     += 1

        
        elapsed = time.time() - t0
        fps_now = 1.0 / max(elapsed, 1e-6)
        stats['fps_list'].append(fps_now)

        cv2.rectangle(frame, (0, 0), (220, 60), (0, 0, 0), -1)
        cv2.putText(frame, f'Frame : {frame_idx:>5}',
                    (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        cv2.putText(frame, f'Active: {active_ids:>5}',
                    (8, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 100), 1)
        cv2.putText(frame, f'FPS   : {fps_now:>5.1f}',
                    (8, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 200, 255), 1)

        out.write(frame)

        
        if frame_idx in (0, 30, 60, 120, 200):
            preview_frames.append((frame_idx, frame.copy()))

        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"   … processed {frame_idx} frames | "
                  f"active tracks: {active_ids} | "
                  f"avg FPS: {np.mean(stats['fps_list']):.1f}")

    cap.release()
    out.release()
    print(f"\n✅  Done!  Output saved → {output_path}")
    return stats, preview_frames

In [ ]:
stats, preview_frames = run_tracking(
    VIDEO_PATH, OUTPUT_VIDEO,
    max_frames=MAX_FRAMES,
    skip=SKIP_FRAMES
)

In [ ]:
def show_preview(frames, cols=3):
    n    = len(frames)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax in axes:
        ax.axis('off')
    for ax, (fidx, frm) in zip(axes, frames):
        ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {fidx}', fontsize=10)
        ax.axis('off')
    plt.suptitle('Tracking Preview — Sampled Frames', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

show_preview(preview_frames)

In [ ]:
def plot_stats(stats):
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0d1117')

    title_kw  = dict(color='white', fontsize=12, fontweight='bold', pad=10)
    label_kw  = dict(color='#aaaaaa', fontsize=9)
    tick_kw   = dict(colors='#888888', labelsize=8)

   
    ax1 = fig.add_subplot(2, 3, 1)
    fps_arr = np.array(stats['fps_list'])
    window  = max(1, len(fps_arr) // 50)
    smooth  = np.convolve(fps_arr, np.ones(window)/window, mode='valid')
    ax1.plot(smooth, color='#00cfff', linewidth=1.5)
    ax1.fill_between(range(len(smooth)), smooth, alpha=0.2, color='#00cfff')
    ax1.set_facecolor('#161b22')
    ax1.set_title('Processing FPS', **title_kw)
    ax1.set_xlabel('Frame', **label_kw)
    ax1.set_ylabel('FPS', **label_kw)
    ax1.tick_params(axis='both', **tick_kw)
    ax1.spines[:].set_color('#30363d')

  
    ax2 = fig.add_subplot(2, 3, 2)
    classes = list(stats['class_counts'].keys())
    counts  = [stats['class_counts'][c] for c in classes]
    colors  = [f'#{hash(c) & 0xFFFFFF:06x}' for c in classes]
    bars = ax2.bar(classes, counts, color=colors, edgecolor='#30363d')
    ax2.bar_label(bars, fmt='%d', padding=3, color='white', fontsize=8)
    ax2.set_facecolor('#161b22')
    ax2.set_title('Detection Count by Class', **title_kw)
    ax2.set_ylabel('Total detections', **label_kw)
    ax2.tick_params(axis='both', **tick_kw)
    ax2.spines[:].set_color('#30363d')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

    
    ax3 = fig.add_subplot(2, 3, 3)
    wedges, _, autotexts = ax3.pie(
        counts, labels=None, autopct='%1.1f%%',
        colors=colors, startangle=140,
        wedgeprops=dict(edgecolor='#0d1117', linewidth=1.5),
        textprops=dict(color='white', fontsize=8)
    )
    ax3.legend(wedges, classes, loc='lower center', fontsize=8,
               facecolor='#161b22', labelcolor='white',
               bbox_to_anchor=(0.5, -0.15), ncol=3, framealpha=0.5)
    ax3.set_title('Class Share', **title_kw)

 
    kpis = [
        ('Total Frames',    stats['total_frames']),
        ('Total Detections', stats['total_detections']),
        ('Unique Track IDs', len(stats['unique_ids'])),
        ('Avg FPS',          f"{np.mean(stats['fps_list']):.1f}"),
    ]
     
    ax_kpi = fig.add_subplot(2, 1, 2)
    ax_kpi.set_facecolor('#161b22')
    ax_kpi.axis('off')
    ax_kpi.set_title('Summary Statistics', **title_kw)

    for i, (label, value) in enumerate(kpis):
        x = 0.12 + i * 0.22
        ax_kpi.add_patch(mpatches.FancyBboxPatch(
            (x - 0.08, 0.2), 0.17, 0.55,
            boxstyle='round,pad=0.02',
            facecolor='#21262d', edgecolor='#30363d', linewidth=1.5,
            transform=ax_kpi.transAxes
        ))
        ax_kpi.text(x + 0.005, 0.62, str(value),
                    transform=ax_kpi.transAxes,
                    ha='center', va='center',
                    fontsize=22, fontweight='bold', color='#58a6ff')
        ax_kpi.text(x + 0.005, 0.32, label,
                    transform=ax_kpi.transAxes,
                    ha='center', va='center',
                    fontsize=9, color='#8b949e')

    plt.suptitle('YOLO + DeepSort — Tracking Statistics', color='white',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

plot_stats(stats)

In [ ]:
print("\n" + "="*50)
print("  TRACKING SUMMARY")
print("="*50)
print(f"  Frames processed : {stats['total_frames']}")
print(f"  Total detections : {stats['total_detections']}")
print(f"  Unique track IDs : {len(stats['unique_ids'])}")
print(f"  Average FPS      : {np.mean(stats['fps_list']):.2f}")
print(f"  Class breakdown  :")
for cls, cnt in sorted(stats['class_counts'].items(), key=lambda x: -x[1]):
    print(f"    {cls:<12} {cnt:>6}")
print(f"\n  Output video  → {OUTPUT_VIDEO}")
print("="*50)